# 3D Tomogram Processing with Improved Masking and Viewer
This notebook loads a stack of 2D images, removes background noise using a masking system, normalizes the data, saves it as `.npy`, and visualizes it in 3D with Plotly.

In [1]:
import os
import numpy as np
from skimage import io, transform, filters
import plotly.graph_objects as go

## Parameters

In [2]:
# Folder containing 2D images
input_folder = r'D:\Downloads\Bacterial Flagellar Motors\ML\Val-Test\Val-Test\Images'  # Replace with your folder path
# Output .npy file
output_file = 'volume_tomo_valtest.npy'
# Desired resize dimensions
target_size = (250, 250)
# Thresholding method: 'otsu' or 'manual'
threshold_method = 'otsu'
manual_threshold = 0.1

## Load and resize images

In [3]:
# Load and resize images
file_list = sorted([f for f in os.listdir(input_folder) if f.lower().endswith(('.jpg','.png', '.tif', '.tiff'))])
volume = []

for f in file_list:
    img = io.imread(os.path.join(input_folder, f))
    # Convert to grayscale if needed
    if img.ndim == 3:
        img = img[..., 0]
    # Resize image
    img_resized = transform.resize(img, target_size, anti_aliasing=True, preserve_range=True)
    volume.append(img_resized)

volume = np.stack(volume, axis=0)  # shape: (N, H, W)
print('Volume shape:', volume.shape)

Volume shape: (300, 250, 250)


## Apply masking system (background removal + thresholding)

In [16]:
# Convert to float
vol = volume.astype(np.float32)
# Normalize 0-1
vol = (vol - vol.min()) / (vol.max() - vol.min())

# Determine threshold
if threshold_method == 'otsu':
    base_thresh = filters.threshold_otsu(vol)
    thresh = base_thresh * 0.78  # increase threshold by 20%
else:
    thresh = manual_threshold

# Apply mask (remove background)
mask = vol <= thresh
vol_masked = vol * mask
print('Applied masking with threshold:', thresh)

Applied masking with threshold: 0.40371093


## Save the processed volume as .npy

In [17]:
np.save(output_file, vol_masked)
print('Saved processed volume to', output_file)

Saved processed volume to volume_tomo_valtest.npy
